# Next step: step 4a, disk heat and the starting state
**What it adds:**
- `heat1`, `heat2`: random motion added to each disk's particles. They existed in SPAM but were fixed at 0.
- `sim.initial_state(...)`: the particles at the start of the run, before SPAM takes a step.

**Why both at once:** to check what Wallin's heat actually does. SPAM adds `randm() * heat` to each of vx, vy
and vz, and `randm()` is in [0, 1), so every kick is positive. If that reading is right, a heated disk doesn't
just get random motion: it gets a net velocity of heat/2 in +x, +y and +z, and drifts. This notebook tests that.

**How it connects:**
- **Below, in JSPAM:** only our `simr_interface.f90` changed. Wallin's modules are untouched.
- **Next (4b):** if the drift is real, a `heat_mode` option adds a centered version: the same spread, no drift.
- **Then (4c):** `SimulatorConfig` bundles `n1`, `n2`, `potential`, `heat1`, `heat2` and `heat_mode`.

The full, current demo is `simulator_tour.ipynb`.

**To run it** (from this repo's root, in bash or PowerShell), then open the `127.0.0.1` link it prints:
```bash
docker run --rm -it -p 8888:8888 -v "${PWD}:/gtc" -w /gtc -e UV_PROJECT_ENVIRONMENT=/tmp/venv \
    ogdenm12/gtc-base-image:v2 uv run --group notebook jupyter lab --ip=0.0.0.0 --allow-root --no-browser
```
- `-v "${PWD}:/gtc"` mounts this repo into the container, so edits on Windows show up immediately.
- `UV_PROJECT_ENVIRONMENT=/tmp/venv` keeps the Linux virtual environment inside the container, out of the repo.
  It's rebuilt each time the container starts (a minute or two, mostly compiling SPAM).
- Our package is installed in editable mode. After editing a `.py` file, restart the kernel to pick it up.

## What changed
**JSPAM, `fortran/simr_interface.f90`:** the setup part of `run_collision` moved into `setup_run`, and the clean-up
into `end_run`, so two subroutines can share them. `setup_run` also sets the heat after `STANDARD_GALAXY1/2`
reset it to 0.
```fortran
subroutine run_collision(collision_param, potential, npts1, npts2, heat_1, heat_2, final_state)
  ...
  call setup_run(collision_param, potential, npts1, npts2, heat_1, heat_2)
  nstep_local = int((tend - tstart) / h) + 2
  do i = 1, nstep_local
    call TAKE_A_STEP
  enddo
  final_state = x0
  call end_run
end subroutine run_collision

! New: the same setup, but no steps
subroutine initial_state(collision_param, potential, npts1, npts2, heat_1, heat_2, start_state)
  ...
  call setup_run(collision_param, potential, npts1, npts2, heat_1, heat_2)
  start_state = x0
  call end_run
end subroutine initial_state

subroutine setup_run(...)
  ... ! everything run_collision used to do before the step loop, plus:
  heat1 = heat_1
  heat2 = heat_2
  ...
end subroutine setup_run
```
**Unchanged, Wallin's `profile` in `setup_module.f90`** (where the heat is applied, after the disk is rotated into
the simulation's frame):
```fortran
x0(i,4) = xv3  + randm()*heat
x0(i,5) = yv3  + randm()*heat
x0(i,6) = zv3  + randm()*heat
```
**JSPAM, `src/jspam/__init__.py`:** `run_collision` takes `heat1`, `heat2`, and the new `initial_state` takes the
same arguments.

**`sim.py`:** `simulate` takes `heat1`, `heat2`, and `sim.initial_state` passes through to `jspam.initial_state`.

In [ ]:
%matplotlib inline
import matplotlib.pyplot as plt
import numpy as np

from galactic_tidal_constraints.simr import sim

MODEL_STRING = "-9.93853,-4.5805,15.43348,-1.54025,-2.97899,4.69234,34.41671,29.49607,2.97913,4.2501,301.0,35.5,310.80745,321.98757,0.3,0.3,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0,0,0.24008,-2.41999,5.62715,0.0,1.0,0.0,1.0,0.0,0.0"
params = sim.parse_model_string(MODEL_STRING)
N1, N2 = 2000, 2000
SEED = 1
HEAT = 5

## 1. The starting state
`initial_state` returns the particles before any step. The secondary starts far from where it ends up, because
SPAM begins before the encounter.

In [ ]:
start = sim.initial_state(params, n1=N1, n2=N2, seed=SEED)
end = sim.simulate(params, n1=N1, n2=N2, seed=SEED)

fig, axes = plt.subplots(1, 2, figsize=(12, 6), sharex=True, sharey=True)
for ax, (title, state) in zip(axes, [("start", start), ("end", end)]):
    ax.scatter(state[:N1, 0], state[:N1, 1], s=1, label="primary disk")
    ax.scatter(state[N1:N1 + N2, 0], state[N1:N1 + N2, 1], s=1, label="secondary disk")
    ax.plot(state[-1, 0], state[-1, 1], "kx", ms=12, label="secondary center")
    ax.set_title(title)
    ax.set_aspect("equal")
axes[0].legend(markerscale=5)
plt.show()

## 2. Isolating the kick
Same model, same seed, heat off and heat on. `randm()` is called either way, so the particles get the same
positions and orbital velocities; the only difference is the heat. Subtracting the two velocities gives the
kick SPAM added to each particle.

In [ ]:
cold = sim.initial_state(params, n1=N1, n2=N2, seed=SEED)
hot = sim.initial_state(params, n1=N1, n2=N2, seed=SEED, heat1=HEAT, heat2=HEAT)

print("positions identical:", np.array_equal(hot[:, :3], cold[:, :3]))
kick = hot[:-1, 3:] - cold[:-1, 3:]  # all disk particles, (vx, vy, vz)
for axis, name in enumerate(["vx", "vy", "vz"]):
    k = kick[:, axis]
    print(f"{name}: min {k.min():+.3f}   max {k.max():+.3f}   mean {k.mean():+.3f}   (heat / 2 = {HEAT / 2})")

If the kicks were centered, they'd spread from −heat/2 to +heat/2. Instead every one is between 0 and heat:

In [ ]:
fig, ax = plt.subplots(figsize=(8, 4))
for axis, name in enumerate(["vx", "vy", "vz"]):
    ax.hist(kick[:, axis], bins=50, histtype="step", label=name)
ax.axvline(0, color="k", lw=0.8)
ax.axvline(HEAT / 2, color="k", ls="--", lw=0.8, label="mean, heat / 2")
ax.set_xlabel("velocity added by heat")
ax.set_ylabel("particles")
ax.legend()
plt.show()

## 3. The whole disk moves
Average the velocities of each disk's particles, relative to its galaxy's center. Without heat, the disk's
rotation averages out to about zero. With Wallin's heat, each disk gains about heat/2 in all three directions,
for both galaxies, whatever their orientation.

In [ ]:
def disk_mean_velocity(state, first, last):
    """Mean (vx, vy, vz) of a disk's particles, relative to its galaxy's center."""
    center = np.zeros(3) if first == 0 else state[-1, 3:]  # the primary's center is at rest at the origin
    return (state[first:last, 3:] - center).mean(axis=0)

for title, state in [("heat off", cold), ("heat on ", hot)]:
    print(title, " primary:", disk_mean_velocity(state, 0, N1).round(3),
          "  secondary:", disk_mean_velocity(state, N1, N1 + N2).round(3))

## 4. What that does by the end
Run both to the end. The primary's center stays at the origin, so the mean position of its disk shows whether
the disk has been carried off its own center. With heat on, expect it to be pushed off toward +x, +y, +z
(gravity then pulls it back, so the size depends on the run). Both runs see the same tidal forces, so any
difference between them comes from the drift.

In [ ]:
cold_end = sim.simulate(params, n1=N1, n2=N2, seed=SEED)
hot_end = sim.simulate(params, n1=N1, n2=N2, seed=SEED, heat1=HEAT, heat2=HEAT)

for title, state in [("heat off", cold_end), ("heat on ", hot_end)]:
    print(title, " primary disk's mean position:", state[:N1, :3].mean(axis=0).round(3))

fig, axes = plt.subplots(1, 2, figsize=(12, 6), sharex=True, sharey=True)
for ax, (title, state) in zip(axes, [("heat off", cold_end), (f"heat on ({HEAT})", hot_end)]):
    ax.scatter(state[:N1, 0], state[:N1, 1], s=1)
    ax.scatter(state[N1:N1 + N2, 0], state[N1:N1 + N2, 1], s=1)
    ax.plot(0, 0, "k+", ms=14)
    ax.plot(*state[:N1, :2].mean(axis=0), "ro", ms=6, label="primary disk's mean position")
    ax.set_title(title)
    ax.set_aspect("equal")
axes[0].legend()
plt.show()